In [ ]:
# Section 1: Feature Extraction (Filipino - Calamancy Only)

import pandas as pd
import spacy
from collections import Counter
import re

# Load
df = pd.read_csv("data/final_fil.csv")

try:
    nlp = spacy.load("tl_calamancy_md")
    print("Using Calamancy Tagalog pipeline")
    use_pos = True
except Exception as e:
    print(f"Calamancy not available: {e}.")
    nlp = spacy.blank("tl")
    use_pos = False

features = []

for _, row in df.iterrows():
    text = str(row['sentence'])
    doc = nlp(text)

    if use_pos:
        tokens = [t.text for t in doc if t.is_alpha]
    else:
        tokens = re.findall(r'\b\w+\b', text.lower())

    word_count = len(tokens)
    unique_words = len(set(tokens))

    if use_pos:
        pos_counts = Counter([t.pos_ for t in doc])
        total_tokens = len([t for t in doc if t.is_alpha]) or 1
        noun_pct = pos_counts.get("NOUN", 0) / total_tokens
        verb_pct = pos_counts.get("VERB", 0) / total_tokens
        adj_pct = pos_counts.get("ADJ", 0) / total_tokens
        adv_pct = pos_counts.get("ADV", 0) / total_tokens
    else:
        noun_pct = verb_pct = adj_pct = adv_pct = 0.0

    features.append({
        "sentence": text,
        "label": row["label"],
        "distortion": row["distortion"],
        "language": row["language"],
        "word_count": word_count,
        "unique_words": unique_words,
        "noun_pct": noun_pct,
        "verb_pct": verb_pct,
        "adj_pct": adj_pct,
        "adv_pct": adv_pct
    })

pd.DataFrame(features).to_csv(
    "data/final_outputs/fil_calamancy_raw.csv",
    index=False
)

print("Done.")

In [ ]:
# Section 2: POS & Word Count Analysis (Filipino)
import pandas as pd
from scipy.stats import ttest_ind

# Load
df = pd.read_csv("data/final_outputs/fil_calamancy_raw.csv")

feature_cols = ['word_count', 'noun_pct', 'verb_pct', 'adj_pct', 'adv_pct']

# Split into Distorted vs Non-Distorted based on label
distorted_df = df[df['label'] == 1]
non_distorted_df = df[df['label'] == 0]

print("Distorted count:", len(distorted_df))
print("Non-distorted count:", len(non_distorted_df))

# Compute mean values
mean_distorted = distorted_df[feature_cols].mean()
mean_non_distorted = non_distorted_df[feature_cols].mean()

# Combine into a summary table
summary = pd.DataFrame({
    'mean_distorted': mean_distorted,
    'mean_non_distorted': mean_non_distorted,
})

# Perform t-test
p_values = []
for col in feature_cols:
    _, p = ttest_ind(distorted_df[col], non_distorted_df[col], equal_var=False)
    p_values.append(p)

summary['p_value'] = p_values
summary['significant'] = summary['p_value'] < 0.05
summary['diff'] = summary['mean_distorted'] - summary['mean_non_distorted']

# Sort
summary_sorted = summary.sort_values(by='diff', ascending=False)

# Save
summary_sorted.to_csv("data/final_outputs/fil_pos_distorted_vs_non.csv")

print("Saved Distorted vs Non-Distorted feature summary.")
print(summary_sorted)

In [ ]:
# Section 3: Filipino Empath Feature Extraction
import pandas as pd
from empath import Empath

# Load
df = pd.read_csv("data/final_fil.csv")

# Load custom Filipino Empath categories
custom_csv_path = "data/final_outputs/empath_category_words_fil.csv"
custom_df = pd.read_csv(custom_csv_path)

# Build dictionary
expanded_categories = {}
for _, row in custom_df.iterrows():
    cat = row["category"]
    words = [w.strip() for w in str(row["words"]).split(",")]
    expanded_categories[cat] = words

# Initialize Empath
lexicon = Empath()
for cat, words in expanded_categories.items():
    lexicon.cats[cat] = words

# Feature Extraction
features = []

for _, row in df.iterrows():
    text = str(row['sentence'])

    # Empath analysis
    empath_features = lexicon.analyze(text, categories=list(expanded_categories.keys()), normalize=True)

    row_features = {
        "sentence": text,
        "label": row['label'],
        "distortion": row['distortion'],
        "language": row.get('language', 'fil'),
        **empath_features
    }

    features.append(row_features)

# Save
features_df = pd.DataFrame(features)
features_df.to_csv("data/final_outputs/fil_empath_raw.csv", index=False)

print("Filipino Empath features extracted and saved.")

In [ ]:
# Section 4: Filipino Empath Feature Analysis
import pandas as pd
from scipy.stats import ttest_ind

# Load
df = pd.read_csv("data/final_outputs/fil_empath_raw.csv")

# Identify Empath category columns
exclude_cols = ['sentence', 'label', 'distortion', 'language']
empath_cols = [c for c in df.columns if c not in exclude_cols]

# Split into distorted vs non-distorted based on label
distorted_df = df[df['label'] == 1]
non_distorted_df = df[df['label'] == 0]

# Compute mean values
mean_distorted = distorted_df[empath_cols].mean()
mean_non_distorted = non_distorted_df[empath_cols].mean()

# Combine
summary = pd.DataFrame({
    'mean_distorted': mean_distorted,
    'mean_non_distorted': mean_non_distorted,
})

# Perform t-test
p_values = []
for col in empath_cols:
    _, p = ttest_ind(distorted_df[col], non_distorted_df[col], equal_var=False)
    p_values.append(p)

summary['p_value'] = p_values
summary['significant'] = summary['p_value'] < 0.05
summary['diff'] = summary['mean_distorted'] - summary['mean_non_distorted']

# Sort
summary_sorted = summary.sort_values(by='diff', ascending=False)

# Save
summary_sorted.to_csv("data/final_outputs/fil_empath_distorted_vs_non.csv")

print("Saved")
print(summary_sorted)

In [ ]:
# Section 5: Top Filipino Empath Categories per Distortion Type
import pandas as pd

# Load
df = pd.read_csv("data/final_outputs/fil_empath_raw.csv")

# Identify Empath columns
exclude_cols = ['sentence', 'label', 'distortion', 'language']
empath_cols = [c for c in df.columns if c not in exclude_cols]

# Filter only distorted texts)
distorted_df = df[df['label'] == 1]

#Average category scores per distortion type
distortion_types = distorted_df['distortion'].unique()
per_type_summary = {}

for dtype in distortion_types:
    per_type_summary[dtype] = distorted_df[distorted_df['distortion'] == dtype][empath_cols].mean()

per_type_df = pd.DataFrame(per_type_summary).transpose()

# Find top N categories per distortion type
top_n = 5
top_categories_per_type = {}

for dtype in per_type_df.index:
    sorted_categories = per_type_df.loc[dtype].sort_values(ascending=False)
    top_categories_per_type[dtype] = sorted_categories.head(top_n)

# Save
top_categories_df = pd.DataFrame(top_categories_per_type)
top_categories_df.to_csv("data/final_outputs/fil_empath_categories_per_distortion.csv")

print("Saved")
print(top_categories_df)